In [3]:
import pandas as pd
import json
import Go_tools

In [4]:
gobasic = json.load(open("/Users/michael/Data/Reference_data/go-basic.json", "r"))

In [5]:
go_term_matrix = pd.read_csv(
    "/Users/michael/Data/Reference_data/Arabdidopsis_go_matrix_from_tair_gaf.csv",
    index_col=0,
)
go_term_matrix

,GO:0000001,GO:0000009,GO:0000012,GO:0000014,GO:0000023,GO:0000024,GO:0000025,GO:0000026,GO:0000027,GO:0000028,...,GO:2001071,GO:2001080,GO:2001141,GO:2001142,GO:2001143,GO:2001147,GO:2001227,GO:2001280,GO:2001289,GO:2001294
GeneID,,,,,,,,,,,,,,,,,,,,,
AT1G01010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AT5G67600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67610,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
namespace_columns = [
    "biological_process",
    "molecular_function",
    "cellular_component",
]

graph = gobasic["graphs"][0]
records = []

for go_term in graph.get("nodes", []):
    meta = go_term.get("meta", {})
    basic_property_values = meta.get("basicPropertyValues", [])
    namespace_value = None

    for property_value in basic_property_values:
        if property_value.get("pred", "").endswith("#hasOBONamespace"):
            namespace_value = property_value.get("val")
            break

    go_identifier = go_term.get("id", "")
    go_number = go_identifier.rsplit("/", 1)[-1].replace("GO_", "")
    if not go_number.isdigit():
        continue

    formatted_go_id = f"GO:{int(go_number):07d}"
    record = {column: 0 for column in namespace_columns}
    if namespace_value in record:
        record[namespace_value] = 1
    records.append((formatted_go_id, record))

go_namespace_df = pd.DataFrame(
    [record for _, record in records],
    index=[go_id for go_id, _ in records],
)
go_namespace_df.index.name = "go_id"
go_namespace_df = go_namespace_df.sort_index()
go_namespace_df

,biological_process,molecular_function,cellular_component
go_id,,,
GO:0000001,1,0,0
GO:0000002,1,0,0
GO:0000003,1,0,0
GO:0000004,0,0,0
GO:0000005,0,1,0
...,...,...,...
GO:7770072,0,1,0
GO:7770073,0,1,0
GO:7770074,1,0,0


In [7]:
biological_process_go_terms = go_namespace_df[
    go_namespace_df["biological_process"] == 1
].index
molecular_function_go_terms = go_namespace_df[
    go_namespace_df["molecular_function"] == 1
].index

matrix_biological_process = go_term_matrix.loc[
    :, go_term_matrix.columns.isin(biological_process_go_terms)
]

In [8]:
matrix_molecular_function = go_term_matrix.loc[
    :, go_term_matrix.columns.isin(molecular_function_go_terms)
]
matrix_molecular_function

,GO:0000009,GO:0000014,GO:0000026,GO:0000030,GO:0000033,GO:0000035,GO:0000036,GO:0000049,GO:0000062,GO:0000064,...,GO:1990825,GO:1990837,GO:1990841,GO:1990931,GO:2001066,GO:2001070,GO:2001071,GO:2001080,GO:2001147,GO:2001227
GeneID,,,,,,,,,,,,,,,,,,,,,
AT1G01010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AT5G67600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67610,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
matrix_biological_process

,GO:0000001,GO:0000012,GO:0000023,GO:0000024,GO:0000025,GO:0000027,GO:0000028,GO:0000038,GO:0000041,GO:0000045,...,GO:2001008,GO:2001009,GO:2001011,GO:2001057,GO:2001141,GO:2001142,GO:2001143,GO:2001280,GO:2001289,GO:2001294
GeneID,,,,,,,,,,,,,,,,,,,,,
AT1G01010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT1G01046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AT5G67600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67610,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT5G67620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
go_namespace_df.sum(axis=1).max()

np.int64(1)

In [11]:
gene_list = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Junk_data/large_deseq_gene_list.csv",
    header=None,
)

In [12]:
gene_list

,0
0,AT1G03600
1,AT1G03630
2,AT1G03680
3,AT1G06680
4,AT1G07180
...,...
144,AT5G58250
145,AT5G58310
146,AT5G61410
147,AT5G64040


In [13]:
cleaned_raw_transcriptome = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/cleaned_raw_transcriptome.csv",
    index_col=0,
)

In [14]:
background_genes_list = cleaned_raw_transcriptome.columns.tolist()

In [15]:
go_term_names = pd.read_csv(
    "/Users/michael/Data/Reference_data/goterm_names.csv", index_col=0
)
go_term_names

,x
GO:0000001,mitochondrion inheritance
GO:0000006,high-affinity zinc transmembrane transporter a...
GO:0000007,low-affinity zinc ion transmembrane transporte...
GO:0000009,"alpha-1,6-mannosyltransferase activity"
GO:0000010,heptaprenyl diphosphate synthase activity
...,...
GO:7770007,L-arginine conjugated cholate hydrolase activity
GO:7770008,L-histidine conjugated cholate hydrolase activity
GO:7770009,L-tryptophan conjugated cholate hydrolase acti...
GO:7770010,GTPBP3-MTO1 complex


In [16]:
cont_tables_biological_process = Go_tools.generate_contigency_tables(
    go_annotations=matrix_biological_process,
    gene_list=gene_list[0].tolist(),
    use_background_genes=True,
    background_genes=background_genes_list,
)

fishers_results_biological_process = Go_tools.fishers_exact_on_contigency_tables(
    all_go_contigency_tables=cont_tables_biological_process,
    original_GO_term_panda=matrix_biological_process,
)

corrected_fishers_results_biological_process = (
    Go_tools.multi_hypothesis_correct_fishers_exact(fishers_results_biological_process)
)
named_biological_process_results = corrected_fishers_results_biological_process.merge(
    go_term_names, left_index=True, right_index=True, how="left"
)
named_biological_process_results.sort_values("P_value").head(40)

/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  - go_subset_distribution_num_genes_annotatated_with_go_term[i]
/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  go_subset_distribution_num_genes_annotatated_with_go_term[i],
/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positio

,P_value,x
GO:0015979,5.490281e-08,photosynthesis
GO:0009658,7.824630e-07,chloroplast organization
GO:0009409,3.213260e-06,response to cold
GO:0010207,2.091175e-05,photosystem II assembly
GO:0090391,5.246588e-05,granum assembly
GO:0110102,1.013277e-04,ribulose bisphosphate carboxylase complex asse...
GO:0010196,1.217389e-04,nonphotochemical quenching
GO:0032544,2.613395e-04,plastid translation
GO:0006412,3.910690e-04,translation
GO:0009773,2.974495e-03,photosynthetic electron transport in photosyst...


In [17]:
cont_tables_molecular_function = Go_tools.generate_contigency_tables(
    go_annotations=matrix_molecular_function,
    gene_list=gene_list[0].tolist(),
    use_background_genes=True,
    background_genes=background_genes_list,
)

fishers_results_molecular_function = Go_tools.fishers_exact_on_contigency_tables(
    all_go_contigency_tables=cont_tables_molecular_function,
    original_GO_term_panda=matrix_molecular_function,
)

corrected_fishers_results_molecular_function = (
    Go_tools.multi_hypothesis_correct_fishers_exact(fishers_results_molecular_function)
)
named_molecular_function_results = corrected_fishers_results_molecular_function.merge(
    go_term_names, left_index=True, right_index=True, how="left"
)
named_molecular_function_results.sort_values("P_value").head(40)

/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  - go_subset_distribution_num_genes_annotatated_with_go_term[i]
/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  go_subset_distribution_num_genes_annotatated_with_go_term[i],
/Users/michael/Git/Bioinformatics_utils/Utilities/Go_tools.py:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positio

,P_value,x
GO:0003729,1.322906e-19,mRNA binding
GO:1901149,4.867532e-06,salicylic acid binding
GO:0008266,2.727581e-04,poly(U) RNA binding
GO:0003735,1.529514e-03,structural constituent of ribosome
GO:0003959,5.709773e-03,NADPH dehydrogenase activity
GO:0008047,7.225183e-03,enzyme activator activity
GO:0016168,9.434361e-03,chlorophyll binding
GO:0009055,1.081009e-02,electron transfer activity
GO:0042132,1.698718e-02,"fructose 1,6-bisphosphate 1-phosphatase activity"
GO:0048529,1.698718e-02,magnesium-protoporphyrin IX monomethyl ester (...


In [ ]:
named_large_fishers_results.sort_values("P_value")